# Phase 11: YOLOv8 Leaf Detector Export to PT and ONNX

This notebook copies your trained custom weights `best.pt` directly into the gateway models folder (`ai/models/detector/leaf_detector.pt`) and exports the model to **ONNX** format for future cross-platform and edge on-device inference compilation.

In [ ]:
# Setup paths and imports
from pathlib import Path
import sys
import glob
import re
import shutil
from ultralytics import YOLO

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != 'ai' and PROJECT_ROOT.parent != PROJECT_ROOT:
    if (PROJECT_ROOT / 'ai').exists():
        PROJECT_ROOT = PROJECT_ROOT / 'ai'
        break
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import utils

In [ ]:
# Find latest training weights run folder inside outputs/runs/detect/
run_dirs = glob.glob(str(PROJECT_ROOT / "outputs/runs/detect/yolov8n_leaf_detection*")) + \
           glob.glob(str(PROJECT_ROOT.parent / "outputs/runs/detect/yolov8n_leaf_detection*"))

def get_run_index(path_str):
    match = re.search(r'yolov8n_leaf_detection-(\d+)', path_str)
    return int(match.group(1)) if match else 1

if run_dirs:
    run_dirs.sort(key=get_run_index, reverse=True)
    latest_run = Path(run_dirs[0])
    best_weights = latest_run / "weights" / "best.pt"
    print("Exporting from latest folder:", latest_run)
else:
    best_weights = Path("outputs/runs/detect/yolov8n_leaf_detection/weights/best.pt")

Exporting from latest folder: o:\Hackthons\KrishiOS\ai\outputs\runs\detect\yolov8n_leaf_detection-8


In [6]:
# Copy custom best weights to the models folder for uvicorn access
if best_weights.exists():
    dest_weights = utils.paths.DETECTOR_MODEL_DIR / "leaf_detector.pt"
    shutil.copy(best_weights, dest_weights)
    print("✅ Best weights copied to gateway model folder:", dest_weights)
    
    # Export to ONNX
    model_to_export = YOLO(dest_weights)
    onnx_path = model_to_export.export(format="onnx")
    print("✅ YOLO model successfully exported to ONNX format at:", onnx_path)
else:
    print("⚠️ Model weights file best.pt not found on disk. Complete training phase first.")

Exporting from latest folder: o:\Hackthons\KrishiOS\ai\outputs\runs\detect\yolov8n_leaf_detection-8
